In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("../utils"))

from pathlib import Path
import re
import torch as th
import numpy as np
import gymnasium as gym
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.policies import ActorCriticPolicy
import torch.nn as nn
from imitation.util import logger as imit_logger
from torch.utils.data import DataLoader
from imitation.data.types import Transitions
import datetime
from imitation.rewards.reward_nets import BasicRewardNet
from imitation.util.networks import RunningNorm
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv
from final_fight import FinalFightActionWrapper, make_env
from stable_baselines3 import PPO

from final_fight import TemporalAttentionLSTM
import gc
from IPython import get_ipython
import cv2

from utils import get_last_index

current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_path = f"./tensorboard_ppo/run_{current_time}"
custom_logger = imit_logger.configure(log_path, ["tensorboard", "stdout"])

gc.collect()
th.cuda.empty_cache()

demo_path = 'demos/'

train_path = 'ppo/'
sub_train_path = train_path + "steps/"

os.makedirs(train_path, exist_ok=True)
os.makedirs(sub_train_path, exist_ok=True)


ENT_WEIGHT= 1e-2 # 1e-3 # 1e-5 # 1e-3 # 0 # 1e-4 # 0.001 # 0.01 # 0 # 1e-3 # 0 # 1e-4
BATCH_SIZE= 512 # 512 # 128 # 128 # 256 # 64 # 256 # 128 # 64 # 128 # 32 # 64 # 128
LEARNING_RATE= 1e-5 # 1e-6 # 1e-5 # 1e-4 # 4e-4  # 1e-4 # 5e-5 # 1e-4 # 2e-4
NUMBER_TRAIN = 200
N_ENVS=8
N_STEPS=4096

action_space = gym.spaces.MultiBinary(6)

observation_space = gym.spaces.Box(
    low=0,
    high=255,
    shape=(96, 96, 1), # 1 frame 96x96
    dtype=np.uint8,
)


last_index = get_last_index(demo_path, "demos", "pt")

print("last_index: " + str(last_index))


class CustomActorCriticPolicy(ActorCriticPolicy):
    def __init__(self, observation_space, action_space, lr_schedule, *args, **kwargs):

        kwargs.pop("features_extractor_class", None)
        kwargs.pop("features_extractor_kwargs", None)
        kwargs.pop("net_arch", None)
    
        super().__init__(
            observation_space,
            action_space,
            lr_schedule,
            *args,
            **kwargs,
            features_extractor_class=TemporalAttentionLSTM,
            features_extractor_kwargs=dict(features_dim=256),
            net_arch=dict(pi=[256, 256], vf=[256, 256])
        )
        self._last_dones = None
        self.retain_graph = True
    
    def forward(self, obs, deterministic=False):
        if hasattr(self.features_extractor, 'reset_hidden'):
            if self._last_dones is not None:
                self.features_extractor.reset_hidden(self._last_dones)
        
        return super().forward(obs, deterministic)
    
    def predict(self, observation, state=None, episode_start=None, deterministic=False):
        if episode_start is not None:
            self._last_dones = episode_start
        
        return super().predict(observation, state, episode_start, deterministic)


env = make_vec_env(make_env(human=False), n_envs=N_ENVS, vec_env_cls=SubprocVecEnv)

last_index_imitation = get_last_index(str(sub_train_path), "ppo_policy", "zip")

print(f"Last index ppo: {last_index_imitation}")

model = None

if last_index_imitation < 0:
    model = PPO(
        policy=CustomActorCriticPolicy, 
        env=env,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        ent_coef=ENT_WEIGHT,
        n_steps=4096 * N_ENVS,
        verbose=1,
        device="cuda"
    )
    
    print("Starting from BC model")
    loaded_policy = CustomActorCriticPolicy.load(f"./imitation/steps/bc_policy36.zip", device="cuda")
    model.policy.load_state_dict(loaded_policy.state_dict())
else:
    model = PPO.load(
        f"./ppo/steps/ppo_policy{last_index_imitation}.zip", 
        env=env,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        ent_coef=ENT_WEIGHT,
        n_steps=N_STEPS * N_ENVS,
        verbose=1,
        device="cuda"
    )
    print(f"Loading model from ppo number: {last_index_imitation}")

model.set_logger(custom_logger)

current_epoch = last_index_imitation + 1

run_name = f"PPO_FineTuning_{current_time}"


for i in range(NUMBER_TRAIN):
    model.learn(
        total_timesteps=25_000,
        reset_num_timesteps=False,
        tb_log_name=run_name
    )
    model.save(f"./ppo/steps/ppo_policy{current_epoch}.zip")

    current_epoch += 1
    
gc.collect()
del model

th.cuda.empty_cache()

print("Force cell kernel reset")
get_ipython().kernel.do_shutdown(restart=True)

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


last_index: 3
Last index ppo: 63


D:\Python311\Lib\site-packages\stable_baselines3\common\save_util.py:449: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  th_object = th.load(file_content, map_location=device

Wrapping the env in a VecTransposeImage.
Loading model from ppo number: 63
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 2.67e+03 |
|    ep_rew_mean     | 114      |
| time/              |          |
|    fps             | 303      |
|    iterations      | 1        |
|    time_elapsed    | 863      |
|    total_timesteps | 17039360 |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 2.67e+03    |
|    ep_rew_mean          | 119         |
| time/                   |             |
|    fps                  | 298         |
|    iterations           | 1           |
|    time_elapsed         | 878         |
|    total_timesteps      | 17301504    |
| train/                  |             |
|    approx_kl            | 0.005946721 |
|    clip_fraction        | 0.0638      |
|    clip_range           | 0.2         |
|    entropy_loss         | -3.12      